# EDA 08 — Is "allocation rate" a well-defined quantity?

**Question.** EDA 05 established that imputation is prevalent and independent of sampling noise, which is what justifies treating data completeness as its own axis. This notebook asks a prior question about the *measure itself*: when we say "X% of income was allocated," is that a property of the data, or a property of a choice we made without saying so?

**Plain-English setup.** When a household skips a question, the Census Bureau fills the blank with a statistically plausible value — **imputation**, which ACS documentation calls **allocation** (see [`docs/glossary.md`](../docs/glossary.md)). An **allocation rate** is the share of values that were filled in rather than reported.

The trouble is that a rate is a fraction, and a fraction needs a bottom. The Bureau's flag tells us the *numerator* — this value was allocated, yes or no. Nobody supplies the **denominator**. Divide by everyone? By everyone old enough to have income? By everyone who actually has that kind of income? Each is defensible, and this notebook shows they do not agree.

**Why this notebook and not EDA 05.** EDA 05 works with published tract tables, where allocation arrives pre-aggregated and the denominator is already baked in. Here we use **PUMS microdata** — one row per person — where the six income sources are separately flagged and we can choose the denominator ourselves. PUMS is the only place that split exists; the detailed tables do not publish it.

**Reproducibility.** Reads `data/raw/pums_<vintage>_nj_alloc_flags.parquet` from disk. No network, no API key. Regenerate with `python ingestion/pull_pums_alloc_flags.py --replicates`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Notebook lives in /notebooks; the analysis package lives at the repo root.
REPO_ROOT = Path.cwd() if (Path.cwd() / "analysis").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from analysis import alloc_denominator as ad
from analysis import alloc_profile as ap
from analysis import common
from analysis import replicate as rep

VINTAGE = common.ACS_VINTAGE
PUMS = REPO_ROOT / "data" / "raw" / f"pums_{VINTAGE}_nj_alloc_flags.parquet"

if not PUMS.exists():
    raise FileNotFoundError(
        f"{PUMS.relative_to(REPO_ROOT)} not found.\n"
        f"Regenerate with:  python ingestion/pull_pums_alloc_flags.py --replicates"
    )

raw = pd.read_parquet(PUMS)
print(f"ACS 5-year PUMS, vintage {VINTAGE}, New Jersey")
print(f"{len(raw):,} person records x {raw.shape[1]} columns")
print(f"replicate weights present: {rep.has_replicates(raw)} "
      f"({len(rep.available_replicates(raw))} of {rep.N_REPLICATES})")


## 1. Data quality before analysis

Two traps sit in this file, and both are silent — they produce plausible wrong answers rather than errors.

**Trap 1: the N/A codes are not uniform.** Four income variables use `-1` to mean "not applicable," but `SEMP` (self-employment) and `INTP` (interest) use `-10001`, because for those two a value of `-1` is a legitimate **one-dollar loss**. A blanket `< 0` filter would therefore discard real data from exactly the two sources we most want to study. `analysis/alloc_denominator.py` keeps a per-variable code map; the assert below pins it.

**Trap 2: `FHINCP` is a household flag on person rows.** It arrives once per *person* but describes their *household*, so summing person weights over it counts each household once per resident. It is deliberately excluded from the six sources below.

In [ ]:
# Trap 1: the N/A code map is per-variable, not global.
assert ad.NA_CODE["WAGP"] == -1, "wages use -1 for N/A"
assert ad.NA_CODE["SEMP"] == -10001, "self-employment uses -10001 (-1 is a real $1 loss)"
assert ad.NA_CODE["INTP"] == -10001, "interest uses -10001 for the same reason"

# Show the consequence rather than asserting it abstractly: how many records
# would a naive `< 0` filter throw away that are actually real losses?
for amount in ("SEMP", "INTP"):
    na = ad.NA_CODE[amount]
    real_losses = ((raw[amount] < 0) & (raw[amount] != na)).sum()
    print(f"{amount:6} N/A code {na:>7} | records with a REAL negative value "
          f"(a loss): {real_losses:,}")

print()
print("The six income sources, and the amount column each flag refers to:")
for flag, amount in ad.FLAG_TO_AMOUNT.items():
    print(f"  {flag:6} -> {amount:6}  (N/A sentinel {ad.NA_CODE[amount]})")

# Trap 2: FHINCP is household-level and is not one of the six.
assert "FHINCP" not in ad.FLAG_TO_AMOUNT, \
    "FHINCP is a HOUSEHOLD flag arriving on person rows -- not a person-level source"

## 2. What does "allocated" actually mean?

Before comparing rates, notice that the flag is doing two different jobs.

Sometimes allocation means **estimating an amount** — "this person probably earned about \$47,000." Sometimes it means **deciding a category** — "this person receives no public assistance." The published flag cannot tell them apart, but in microdata we can: look at whether the allocated *amount* came out as zero.

In [ ]:
zeros = ad.zero_allocation_share(raw)

print("Share of allocations that imputed a ZERO rather than an amount\n")
print(f"{'source':<8} {'allocated':>11} {'to zero':>11} {'share zero':>11}")
for _, r in zeros.iterrows():
    print(f"{r['source']:<8} {r['allocated']:>11,} {r['allocated_to_zero']:>11,} "
          f"{r['share_zero']:>10.1%}")

z = zeros.set_index("source")["share_zero"]

# Headline claim, guarded: for the rare income types "allocated" overwhelmingly
# means "the Bureau decided you have none of this".
assert z["FPAP"] > 0.95, "public assistance allocations should be almost all zeros"
assert z["FSEMP"] > 0.85, "self-employment allocations should be mostly zeros"
assert z["FWAGP"] < 0.40, "wage allocations should mostly be real amounts"
assert z["FPAP"] > 2 * z["FWAGP"], "the two acts must be clearly separated"

print(f"\nPublic assistance is {z['FPAP'] / z['FWAGP']:.1f}x more likely than wages")
print("to be an imputed zero. These are different acts wearing one flag.")

## 3. Allocation is at least two mechanisms

There is a second thing the pooled flag hides. Some records have **one** income item filled in. Others have **all six** — which is not item-level imputation at all, but the Bureau substituting a whole person's record with a donor's.

If we pool those together, the wholesale substitutions drag every source toward a common rate and make the six look more alike than they are.

In [ ]:
flags = list(ad.FLAG_TO_AMOUNT)
any_flag = (raw[flags] == 1).any(axis=1)
whole = ad.whole_record_mask(raw)

share_whole_of_flagged = whole.sum() / any_flag.sum()
weighted_whole = raw.loc[whole, ad.PERSON_WEIGHT].sum() / raw[ad.PERSON_WEIGHT].sum()

print(f"records with at least one income flag : {any_flag.sum():>9,}")
print(f"records with ALL SIX flags set        : {whole.sum():>9,}")
print(f"  -> share of flagged that are whole  : {share_whole_of_flagged:>9.1%}")
print(f"  -> weighted share of all persons    : {weighted_whole:>9.1%}")

assert share_whole_of_flagged > 0.35, \
    "whole-record substitution should be a large minority of all flagged records"

# What does pooling cost us? Compare the spread across the six sources with and
# without the whole-record rows, holding the definition fixed (D2/S1).
swept_all = ad.sweep(raw)
swept_item = ad.sweep(raw.loc[~whole])
DEFN = "D2 uni15+/S1 all"


def fold(swept: pd.DataFrame) -> float:
    r = swept.loc[swept["definition"] == DEFN, "rate"]
    return r.max() / r.min()


print(f"\nSpread across the six sources under {DEFN}:")
print(f"  pooled (as published)          : {fold(swept_all):.2f}x")
print(f"  item-level only (whole removed): {fold(swept_item):.2f}x")
print("\nThe sources were never similar. The wholesale substitutions were")
print("flattening the picture.")

## 4. The eight defensible definitions

Four denominators, two scopes, all combinations. None of these is a trick — each answers a real question, and each is the obvious choice from some point of view.

In [ ]:
print("DENOMINATORS\n")
for k, v in ad.DENOMINATOR_NOTES.items():
    print(f"  {k}\n      {v}\n")
print("SCOPES (orthogonal to the above)\n")
for k, v in ad.SCOPE_NOTES.items():
    print(f"  {k}\n      {v}\n")

swept = swept_all
rates = ad.rate_table(swept)
ranks = ad.rank_table(swept)

assert rates.shape == (6, 8), "six sources x eight definitions"
assert (rates.stack().dropna().between(0, 1)).all(), "every rate must be a proportion"

print("ALLOCATION RATE (%) UNDER EVERY DEFINITION\n")
(rates * 100).round(2)

## 5. Do the definitions agree?

Now the point of the exercise. If all eight definitions ranked the six income sources the same way, the denominator would be a presentational detail. Read the heat map below by **column**: each column is one definition's ordering, and if the definitions agreed, every row would be a flat band of colour.

**Rank volatility** is the gap between a source's best and worst rank across definitions. A swing equal to *(number of sources − 1)* means that source can be ranked anywhere at all.

In [ ]:
vol = ad.volatility(swept)

print(f"{'source':<8} {'best':>5} {'worst':>6} {'swing':>6}   rate range")
for _, r in vol.iterrows():
    print(f"{r['source']:<8} {r['best_rank']:>5} {r['worst_rank']:>6} "
          f"{r['rank_swing']:>6}   {r['rate_min']:6.2%} - {r['rate_max']:6.2%}"
          f"  ({r['rate_fold']:5.1f}x)")

s = ad.summary(swept)
print(f"\nLargest rank swing: {s['max_rank_swing']} of {s['n_sources']} places")

assert s["max_rank_swing"] >= s["n_sources"] - 1, (
    "at least one source should be rankable anywhere -- if this fails, the "
    "denominator claim has weakened and the write-up needs revisiting"
)
worst = vol.iloc[0]
print(f"-> {worst['source']} ranks anywhere from {worst['best_rank']} to "
      f"{worst['worst_rank']}. Its ordering is a property of the definition,")
print("   not of the data.")

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Chart styling. Single-hue encoding throughout: every chart below shows ONE
# measure, so colour carries magnitude, not identity -- no legend needed and no
# categorical palette to validate. Values are the validated sequential blue ramp.
INK        = "#0b0b0b"
INK_MUTED  = "#52514e"
SURFACE    = "#fcfcfb"
BLUE_400   = "#3987e5"
BLUE_450   = "#2a78d6"
BLUE_RAMP  = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec",
              "#5598e7", "#3987e5", "#2a78d6", "#256abf", "#1c5cab",
              "#184f95", "#104281", "#0d366b"]
SEQ_CMAP   = mpl.colors.LinearSegmentedColormap.from_list("blue_seq", BLUE_RAMP)

mpl.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": INK_MUTED,
    "axes.labelcolor": INK,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "text.color": INK,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "grid.color": "#e6e5e1",
    "grid.linewidth": 0.8,
    "font.size": 9,
    "figure.dpi": 110,
})

# Rank is ordered magnitude (1 = most allocated), so a single-hue sequential
# ramp is the correct encoding: darker = more allocated. Values are printed in
# every cell, so the colour is reinforcement rather than the only channel.
order = vol["source"].tolist()               # most volatile first
# rank_table carries object dtype (ranks are whole numbers); imshow needs float.
plot_ranks = ranks.loc[order].astype(float)

fig, ax = plt.subplots(figsize=(8.4, 3.2))
im = ax.imshow(plot_ranks.values, cmap=SEQ_CMAP.reversed(),
               vmin=1, vmax=plot_ranks.values.max(), aspect="auto")

ax.set_xticks(range(plot_ranks.shape[1]))
ax.set_xticklabels(plot_ranks.columns, rotation=38, ha="right", fontsize=8)
ax.set_yticks(range(plot_ranks.shape[0]))
ax.set_yticklabels(plot_ranks.index, fontsize=9)
ax.set_xlabel("denominator / scope definition")

# Direct labels in every cell; ink colour flips on the dark end for contrast.
for i in range(plot_ranks.shape[0]):
    for j in range(plot_ranks.shape[1]):
        v = plot_ranks.values[i, j]
        ax.text(j, i, str(int(v)), ha="center", va="center", fontsize=8.5,
                color="#ffffff" if v <= 2 else INK)

# 2px surface gap between cells, per the mark spec.
ax.set_xticks(np.arange(-0.5, plot_ranks.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, plot_ranks.shape[0], 1), minor=True)
ax.grid(which="minor", color=SURFACE, linewidth=2)
ax.grid(which="major", visible=False)
ax.tick_params(which="minor", length=0)

ax.set_title("Rank of each income source (1 = most allocated)\n"
             "Flat rows would mean the definitions agree. They do not.",
             loc="left", pad=10)
plt.tight_layout()
plt.show()

### Pairwise agreement

**Kendall tau** measures whether two orderings agree: `+1` identical, `0` unrelated, `−1` exactly reversed. Add it to [`docs/glossary.md`](../docs/glossary.md) if it is not there yet. It is computed in `analysis/alloc_denominator.py` rather than imported, because `pandas.corr(method="kendall")` delegates to `scipy`, which is deliberately not a project dependency.

A negative tau between two definitions means they do not merely disagree on size — they put the sources in **opposite order**.

In [ ]:
agr = ad.agreement(swept)

print(f"comparable pairs         : {s['n_pairs_comparable']} of {s['n_pairs']}")
print(f"pairs that REVERSE order : {s['n_pairs_reversed']}")
print(f"mean Kendall tau         : {s['mean_tau']:+.2f}")
print(f"worst pair               : tau = {s['min_tau']:+.2f}")
w = agr.iloc[0]
print(f"                           {w['def_a']}  vs  {w['def_b']}")

assert s["mean_tau"] < 0.6, "definitions should be far from interchangeable"
assert s["n_pairs_reversed"] > 0, "at least some pairs should reverse the ordering"

print("\nCAUTION -- the reversed-pair COUNT is not stable between vintages")
print("(12 of 28 at 2024, 8 of 28 at 2023). Quote the maximum rank swing,")
print("which held at 5 in both. See WORKLOG.md.")

## 6. What this means

Three findings, in decreasing order of how comfortable I am defending them.

**1. "Allocated" is two different acts sharing one flag.** For rare income types, allocation almost always means *the Bureau decided you have none of this* — not that it estimated an amount. This is a direct count in microdata with no denominator and no model, so there is very little to argue with.

**2. Whole-record substitution masks real differences between income sources.** Also a direct count. Pooling wholesale substitutions with item-level fills compresses the spread across sources by roughly threefold.

**3. "The allocation rate" is not a well-defined quantity.** Under eight defensible definitions, at least one income source can be ranked anywhere from first to last. This held at both vintages tested.

**The practical rule adopted for the project: never quote a bare allocation rate.** State the definition, or report the sweep. Any downstream quantity — including a composite reliability score — silently inherits one of these choices, and swapping definitions can reverse the conclusion.

**Honest limitation.** The eight definitions are *our* construction, not the Bureau's. Someone could reasonably argue we chose eight that maximise disagreement. Each has a written justification in `ad.DENOMINATOR_NOTES`, but this is a fair challenge to have an answer ready for.

Full write-up: [`docs/allocation-denominator-sensitivity.md`](../docs/allocation-denominator-sensitivity.md), regenerated by `report_alloc_denominators.py`.